# 🎯 Beamforming Theory

A beginner-friendly walkthrough of **Delay-and-Sum Beamforming** with visualizations and code.

## 📚 1. What is Beamforming?
Beamforming is a technique used in sonar, radar, and wireless systems to **focus a sensor array's sensitivity** in a particular direction.

It uses **constructive and destructive interference** to emphasize signals coming from one direction and suppress others.

## 🧮 2. Delay-and-Sum Principle
Each element of a sensor array receives a signal at slightly different times depending on the **Direction of Arrival (DOA)**.

By adding appropriate time delays to each element, we can **align the signals** and sum them constructively:

\[ y(t) = \sum_{n=0}^{N-1} x_n(t + \tau_n) \]

where \( \tau_n \) is the delay for sensor \( n \).

## 🛠️ 3. Setup Simulation Parameters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Signal and array parameters
fs = 48000
duration = 0.01
f_start = 2000
f_end = 6000
array_length = 8
doa_deg = 30  # Direction of arrival in degrees

# Time axis and chirp
t = np.linspace(0, duration, int(fs * duration), endpoint=False)
k = (f_end - f_start) / duration
chirp = np.cos(2 * np.pi * (f_start * t + 0.5 * k * t**2))

## 🧩 4. Create Array Signals with DOA Delays

In [ ]:
def create_array_signals(signal, fs, num_elements, doa_deg, d=0.015, c=1500):
    signals = []
    theta_rad = np.radians(doa_deg)
    for n in range(num_elements):
        delay = (n * d * np.sin(theta_rad)) / c
        delay_samples = int(np.round(delay * fs))
        delayed = np.roll(signal, delay_samples)
        signals.append(delayed)
    return np.array(signals)

signals = create_array_signals(chirp, fs, array_length, doa_deg)

## 📊 5. Plot Simulated Signals from Array

In [ ]:
plt.figure(figsize=(10, 5))
for i in range(array_length):
    plt.plot(t, signals[i] + i, label=f"Sensor {i}")
plt.title("Simulated Array Signals with Delay")
plt.xlabel("Time [s]")
plt.ylabel("Amplitude (offset for clarity)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 🧠 6. Delay-and-Sum Beamforming

In [ ]:
def delay_and_sum(signals, fs, angles_deg, d=0.015, c=1500):
    num_elements, num_samples = signals.shape
    response = []

    for angle in angles_deg:
        theta_rad = np.radians(angle)
        delays = np.array([(n * d * np.sin(theta_rad)) / c for n in range(num_elements)])
        delay_samples = np.round(delays * fs).astype(int)

        aligned_signals = np.zeros(num_samples)
        for i in range(num_elements):
            aligned_signals += np.roll(signals[i], -delay_samples[i])

        response.append(np.max(np.abs(aligned_signals)))  # Use peak response

    return np.array(response)

angles = np.linspace(-90, 90, 181)
response = delay_and_sum(signals, fs, angles)

## 📈 7. Plot Beam Pattern

In [ ]:
import matplotlib.ticker as ticker

plt.figure(figsize=(8, 4))
plt.plot(angles, 20 * np.log10(response / np.max(response) + 1e-12))
plt.title("Beamforming Array Pattern (dB)")
plt.xlabel("Angle (degrees)")
plt.ylabel("Normalized Response (dB)")
plt.grid(True)
plt.gca().xaxis.set_major_locator(ticker.MultipleLocator(15))
plt.tight_layout()
plt.show()

## ✅ Summary
- Delay-and-sum aligns signals by direction
- Focuses sensitivity in chosen DOA
- Works well for uniform linear arrays (ULAs)

This is the foundation of all sonar beamforming systems.